In [5]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
import datetime

llm = ChatGroq(
    api_key="Your-Grok-key",
    model="llama-3.3-70b-versatile"
)

@tool
def get_date() -> str:
    """Returns today's date"""
    return str(datetime.date.today())

@tool
def calculator(expression: str) -> str:
    """Evaluates a math expression like 2+2 or 10*5"""
    return str(eval(expression))

@tool
def ask_resume(question: str) -> str:
    """Answers questions about Neha's resume and experience"""
    return """Neha Kanwadiya, IIT Bombay Engineering Physics 2027.
    Internships: GradPipe (Software Engineering), ISB Hyderabad (Research).
    Projects: AI Notes Generator, YouTube Clone, Heart Disease Model, SplitNow.
    Skills: React, Python, Django, Node.js, TypeScript, ML, LLMs."""

tools = [get_date, calculator, ask_resume]

# Create agent — new way using langgraph
agent = create_react_agent(llm, tools)

def chat(user_input):
    response = agent.invoke({
        "messages": [HumanMessage(content=user_input)]
    })
    return response['messages'][-1].content

# Test
print("=== Test 1: Date ===")
print(chat("What is today's date?"))

print("\n=== Test 2: Math ===")
print(chat("What is 1234 * 5678?"))

print("\n=== Test 3: Resume ===")
print(chat("What internships has Neha done?"))

C:\Users\kanwa\AppData\Local\Temp\ipykernel_1152\4112366434.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


=== Test 1: Date ===
Today's date is May 7, 2026.

=== Test 2: Math ===
The result of 1234 * 5678 is 7006652.

=== Test 3: Resume ===
Neha has done internships at GradPipe as a software engineer and at ISB Hyderabad as a research intern.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from sentence_transformers import SentenceTransformer
import chromadb
from pypdf import PdfReader
import datetime

llm = ChatGroq(api_key="Your-Grok-Key", model="llama-3.3-70b-versatile")

# Load RAG
reader = PdfReader("Neha_Resume.pdf")
full_text = "".join(page.extract_text() for page in reader.pages)
chunks = [full_text[i:i+350] for i in range(0, len(full_text), 350) if full_text[i:i+350].strip()]

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks).tolist()

db = chromadb.Client()
collection = db.get_or_create_collection("resume_day3")
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"c{i}" for i in range(len(chunks))]
)
print("RAG ready")

# Tools — more explicit descriptions to avoid Groq tool call errors
@tool
def search_resume(question: str) -> str:
    """Search Neha's resume. Input must be a specific keyword or phrase.
    Examples: 'ISB internship details', 'GradPipe experience', 'Python skills', 'projects built'"""
    try:
        q_embed = embed_model.encode([question]).tolist()
        results = collection.query(query_embeddings=q_embed, n_results=3)
        return "\n".join(results['documents'][0])
    except Exception as e:
        return f"Could not search resume: {str(e)}"

@tool
def get_date() -> str:
    """Returns today's date. No input needed."""
    return str(datetime.date.today())

@tool
def calculator(expression: str) -> str:
    """Calculates math. Input must be a valid expression like '1234 * 5678' or '100 / 4'"""
    try:
        return str(eval(expression))
    except:
        return "Invalid math expression"

tools = [search_resume, get_date, calculator]

# Memory
memory = MemorySaver()
agent = create_react_agent(
    llm,
    tools,
    checkpointer=memory,
    prompt=SystemMessage(content="""You are an intelligent assistant for 
    Neha Kanwadiya, IIT Bombay Engineering Physics student (2027).
    When asked about her experience, skills, or projects always use 
    search_resume tool with specific keywords.
    For vague follow-up questions like 'tell me more about that', 
    expand on your previous answer using already retrieved information.
    Be concise and helpful.""")
)

config = {"configurable": {"thread_id": "neha_session_1"}}

def chat(user_input):
    try:
        response = agent.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )
        return response['messages'][-1].content
    except Exception as e:
        return f"Error: {str(e)} — try rephrasing your question"

# Interactive chat
print("Neha's AI Assistant — Tools + RAG + Memory")
print("Type 'quit' to exit\n")

while True:
    user = input("You: ")
    if user.lower() == 'quit':
        break
    print(f"\nAI: {chat(user)}\n")
    print("─" * 50 + "\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\kanwa\AppData\Local\Temp\ipykernel_1152\560703857.py:59: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


RAG ready
Neha's AI Assistant — Tools + RAG + Memory
Type 'quit' to exit



You:  tell me about ISB



AI: Error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_resume{"question": "ISB internship details"}</function>'}} — try rephrasing your question

──────────────────────────────────────────────────



...


AI: Neha's skills in software engineering, AI-driven talent marketplace, and experience with Large Language Models (LLMs) and Prompt Engineering make her a strong fit for roles such as:

1. Software Engineer: With her experience in building backend features for a talent marketplace, Neha can excel in software engineering roles that involve designing, testing, and deploying product improvements.
2. AI/ML Engineer: Her experience with LLMs and Prompt Engineering makes her a suitable candidate for AI/ML engineering roles that involve developing and deploying AI models.
3. Data Analyst: Neha's experience in analyzing GitHub activity, project repositories, and developer profiles for hiring insights can be valuable in data analyst roles that involve working with data to inform business decisions.
4. Product Manager: With her experience in collaborating with a fast-paced startup team to design, test, and deploy product improvements, Neha can excel in product management roles that involve ove

You:  Is Quant is suitable for her?



AI: Based on the search results, it appears that Neha has a strong foundation in programming languages such as C++, Python, JavaScript, and HTML/CSS. She also has experience with libraries and frameworks like React.js, Node.js, Flask, TensorFlow, and Keras. Additionally, she has worked on AI-driven projects, including a talent marketplace that analyzes GitHub activity and developer profiles.

However, the search results do not specifically mention quantitative skills or experience with quantitative finance. While Neha's technical skills are impressive, it is unclear whether she has the specific skills or knowledge required for a career in quantitative finance.

To determine whether quant is suitable for Neha, it would be helpful to know more about her interests, skills, and experiences. If she has a strong foundation in mathematics, statistics, and computer science, and is interested in finance, then a career in quantitative finance may be a good fit. However, if she lacks experience 

You:  Is Fintech job she can do?



AI: Based on the search results, it appears that Neha has a strong foundation in software engineering, AI-driven technologies, and experience with Large Language Models (LLMs) and Prompt Engineering. She has also worked on a talent marketplace that connects students with startups and global companies.

While the search results do not specifically mention fintech skills or experience, Neha's technical skills and experience in software engineering and AI-driven technologies could be transferable to a fintech role. Fintech companies often use AI, machine learning, and data analytics to develop financial products and services, so Neha's skills in these areas could be valuable.

Additionally, Neha's experience working with a startup and collaborating with a fast-paced team to design, test, and deploy product improvements could be beneficial in a fintech company, where innovation and adaptability are often key.

However, to determine whether a fintech job is a good fit for Neha, it would be